# 00 · First look at every file

What each Weavve file holds before anything is cleaned. For every file, its size, each column's type,
how many cells are blank, how many different values a column has, one example value, the date range,
and a few sample rows. Nothing here changes the data.

**Before running**, put the Weavve files in `data/` (see `data/README.md`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # run from the notebooks folder

import pandas as pd
from weavve import load, paths
from weavve.load import MARKETPLACE, WEBSITE
pd.set_option("display.max_columns", 50)
print("data:", paths.DATA)
print("outputs:", paths.OUT)

In [ ]:
def structure(df):
    """One row per column. Its type, how many cells are blank, how many different values, and one example."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "blank": df.isna().sum(),
        "blank_%": (df.isna().mean() * 100).round(1),
        "different_values": df.nunique(),
        "example": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })


def date_range(df, col):
    d = pd.to_datetime(df[col], errors="coerce")
    print(f"{col}: {d.min():%d %b %Y} to {d.max():%d %b %Y}, {d.isna().sum()} blank or unreadable")

## All files at a glance

In [ ]:
t = load.orders(); s = load.skus(); e = load.discounts(); f = load.loyalty()
a = load.ads(); neg = load.ads_negatives(); sem = load.semrush()
books, lockfiles = load.gsc_workbooks()

pd.DataFrame([
    ("1 Search Console", "workbooks", len(books), None),
    ("2 Semrush", "rows, all months stacked", len(sem), sem.shape[1]),
    ("3 Orders", "order lines", len(t), t.shape[1]),
    ("4 Products", "products", len(s), s.shape[1]),
    ("8 Discounts", "discount rows", len(e), e.shape[1]),
    ("9.1 Loyalty", "redemptions", len(f), f.shape[1]),
    ("12 Google Ads", "keyword rows", len(a), a.shape[1]),
], columns=["file", "one row is", "rows", "columns"])

## File 3, orders

One row per order line. Sales, refunds and cancellations are mixed in the same table.

In [ ]:
print(t.shape)
date_range(t, "Day")
structure(t)

In [ ]:
t.head()

In [ ]:
print(t["Order sales channel"].value_counts(), "\n")
print(t["Order or sales reversal"].value_counts(), "\n")
print("Cancelled order lines:", int(t["Is canceled order"].sum()))

## File 4, products

In [ ]:
print(s.shape)
structure(s)

In [ ]:
s.head()

## File 8, discounts

In [ ]:
print(e.shape)
structure(e)

In [ ]:
e.head()

## File 9.1, loyalty redemptions

In [ ]:
print(f.shape)
structure(f)

In [ ]:
f.head()

## File 12, Google Ads keywords

Past keyword results, and the words Weavve blocked before.

In [ ]:
print(a.shape, "keyword rows,", len(neg), "blocked words")
structure(a)

In [ ]:
a.head()

## File 2, Semrush organic keywords

One csv per month, stacked into one table. The month comes from the file name.

In [ ]:
print(sem.shape)
print("Months:", sem["month"].min().date(), "to", sem["month"].max().date(), "|", sem["month"].nunique(), "files")
structure(sem)

In [ ]:
sem.head()

## File 1, Search Console

A folder of monthly workbooks. Each workbook has several sheets.

In [ ]:
print(len(books), "workbooks,", lockfiles, "Excel lock files skipped")
first = pd.ExcelFile(books[0])
print("Sheets in", Path(books[0]).name, ":", first.sheet_names)
for sh in first.sheet_names:
    print(f"  {sh}: {first.parse(sh).shape}")

In [ ]:
clicks = load.gsc_site_clicks()
print(clicks.shape)
clicks.sort_values("month").head(10)